# 工作台计算适配验证

验证公开接口与教材/手算锚点，全部数据内存构造，不连接数据库。
教材第1章pp.2–16；VaR例1.3/1.4、正态ES与历史尾部平均。


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from scipy.stats import norm
root = Path.cwd() if (Path.cwd() / 'controller').exists() else Path.cwd().parent
sys.path.insert(0, str(root))
from controller.workbench_calculations import catalog, examples, example_frame, prepare_frame, calculate_frame
assert len(catalog()['chapters']) == 18
assert sum(c['available'] for c in catalog()['chapters']) == 1
assert {o['id'] for o in catalog()['operations']} == {'var','spectral','standard_error','diagnostics'}
def run(df, metadata, operation, **parameters):
    result = calculate_frame(df, metadata, {'operation':operation,'parameters':parameters})
    json.dumps(result, allow_nan=False)
    return result
def value(result):
    return result['metrics'][0]['value']
print('目录与公开接口加载通过')

目录与公开接口加载通过


In [2]:
# 教材例1.3：收益率mean=.1,std=.25；例1.4：对数收益率mean=.05,std=.2。
df, meta = example_frame('textbook_normal')
r = run(df, meta, 'var', model='normal', target_column='arithmetic_return', confidence_level=.95)
np.testing.assert_allclose(value(r), -.1 + .25 * norm.ppf(.95), rtol=1e-12)
assert round(value(r), 3) == .311  # 教材粗略z表数值约.311。
gdf, gm = example_frame('textbook_lognormal')
gr = run(gdf, gm, 'var', model='lognormal', target_column='geometric_return', confidence_level=.95)
np.testing.assert_allclose(value(gr), 1-np.exp(.05-.2*norm.ppf(.95)), rtol=1e-12)
assert abs(value(gr) - .24344) < .00001  # 以精确正态分位点核对公式，不沿用教材粗略取整。
es = run(df, meta, 'spectral', model='normal', target_column='arithmetic_return', weight='es', confidence_level=.95)
np.testing.assert_allclose(value(es), -.1+.25*norm.pdf(norm.ppf(.95))/.05, rtol=1e-7)
print('教材参数法VaR与正态ES独立闭式锚点通过')

教材参数法VaR与正态ES独立闭式锚点通过


In [3]:
# 手算：五个损益[-4,-2,0,1,3]，80% VaR=2，80% ES=4。
h = pd.DataFrame({'x':[-4.,-2.,0.,1.,3.]})
hm = {'fields':{'x':{'kind':'pnl','unit':'amount','sign':'profit'}}}
np.testing.assert_allclose(value(run(h,hm,'var',target_column='x',confidence_level=.8)),2)
np.testing.assert_allclose(value(run(h,hm,'spectral',target_column='x',confidence_level=.8)),4)
losses = -h
lm = {'fields':{'x':{'kind':'loss','unit':'amount','sign':'loss'}}}
np.testing.assert_equal(value(run(losses,lm,'var',target_column='x',confidence_level=.8)),2)
# 历史ES积分在非整数尾部也保留边界观测部分权重。
np.testing.assert_allclose(value(run(h,hm,'spectral',target_column='x',confidence_level=.7)),(4+.5*2)/1.5)
print('历史VaR、ES与损失正负号通过')

历史VaR、ES与损失正负号通过


In [4]:
# 准备链：从价格与现金流派生；元数据和原始样本不被修改。
p, pm = example_frame('demo_prices')
original = p.copy(deep=True)
meta_before = json.dumps(pm)
prepared, metadata, changes = prepare_frame(p, pm, {'date_column':'date','sort_by':['date'],
    'derive':[{'operation':'pnl'},{'operation':'pvpnl'},{'operation':'fvpnl'},
              {'operation':'arithmetic_return'},{'operation':'geometric_return'}]})
pd.testing.assert_frame_equal(p,original)
assert json.dumps(pm) == meta_before
assert prepared.iloc[0][['pnl','arithmetic_return','geometric_return']].isna().all()
assert changes['dropped_rows'] == 0
clean, cm, changed = prepare_frame(prepared,metadata,{'dropna_columns':['arithmetic_return','geometric_return']})
assert changed['dropped_rows'] == 1
comparison = run(clean,cm,'var',model='compare',arithmetic_column='arithmetic_return',geometric_column='geometric_return')
assert len(comparison['metrics']) == 3
np.testing.assert_allclose(clean['geometric_return'], np.log1p(clean['arithmetic_return']))
print('派生准备、首行缺失保留、显式删除及同样本比较通过')

派生准备、首行缺失保留、显式删除及同样本比较通过


In [5]:
# 百分数与损失符号转换必须显式执行且不能重复。
pct = pd.DataFrame({'r':[-2.,1.,3.,-4.]})
pct_meta = {'fields':{'r':{'kind':'arithmetic_return','unit':'percent','sign':'loss'}}}
converted, converted_meta, _ = prepare_frame(pct,pct_meta,{'conversions':[
    {'column':'r','kind':'percent_to_decimal'},{'column':'r','kind':'loss_to_profit'}]})
np.testing.assert_equal(converted.r.to_numpy(), [.02,-.01,-.03,.04])
assert converted_meta['fields']['r'] == {'kind':'arithmetic_return','unit':'decimal','sign':'profit'}
print('显式单位和正负号转换通过')

显式单位和正负号转换通过


In [6]:
# 分位数标准误独立锚点及三类分布的标准误路径。
df, meta = example_frame('textbook_normal')
qse = run(df,meta,'standard_error',method='quantile',model='normal',target_column='arithmetic_return',confidence_level=.95,interval_confidence=.90)
expected = np.sqrt(.95*.05/(3*(norm.pdf(norm.ppf(.95))/.25)**2))
np.testing.assert_allclose(qse['metrics'][1]['value'],expected,rtol=1e-12)
rdf, rm = example_frame('demo_returns')
small = rdf.iloc[:30]
for model,column in [('historical','arithmetic_return'),('normal','arithmetic_return'),('lognormal','geometric_return')]:
    for method in ['bootstrap','quantile','spectral']:
        result=run(small,rm,'standard_error',model=model,method=method,target_column=column,n_resamples=30,n_slices=20)
        assert result['metrics'][1]['value'] >= 0
    for weight in ['es','exponential']:
        for method in ['integral','slices']:
            result=run(small,rm,'spectral',model=model,weight=weight,method=method,target_column=column,n_slices=25)
            assert np.isfinite(value(result))
first=run(small,rm,'standard_error',target_column='arithmetic_return',n_resamples=40,seed=123)
second=run(small,rm,'standard_error',target_column='arithmetic_return',n_resamples=40,seed=123)
assert first == second
print('标准误独立锚点、全部模型组合、Bootstrap可复现通过')

标准误独立锚点、全部模型组合、Bootstrap可复现通过


In [7]:
# 诊断使用全部样本；QQ坐标表、空尾部NaN可用JSON表示。
rdf, rm = example_frame('demo_returns')
result=run(rdf,rm,'diagnostics',target_column='arithmetic_return')
assert result['metrics'][0]['value'] == len(rdf)
assert len(result['charts']) == 4
assert len(result['tables'][2]['rows']) == len(rdf)
print('分布诊断和JSON图表结果通过')

分布诊断和JSON图表结果通过


In [8]:
# 不符合口径的请求必须在调用边界拒绝，而不是静默清洗。
def rejects(action):
    try: action()
    except (ValueError,KeyError): return
    raise AssertionError('Expected explicit boundary error')
rdf, rm = example_frame('demo_returns')
missing=rdf.copy(); missing.loc[0,'arithmetic_return']=np.nan
rejects(lambda:run(missing,rm,'var',target_column='arithmetic_return'))
rejects(lambda:run(rdf,rm,'var',model='normal',target_column='geometric_return'))
rejects(lambda:run(rdf,rm,'var',target_column='arithmetic_return',confidence_level=1))
rejects(lambda:run(rdf,rm,'standard_error',target_column='arithmetic_return',n_resamples=1))
rejects(lambda:run(pct,pct_meta,'var',target_column='r'))
bad=rdf.copy(); bad.loc[0,'geometric_return']+=.01
rejects(lambda:run(bad,rm,'var',model='compare',arithmetic_column='arithmetic_return',geometric_column='geometric_return'))
rejects(lambda:prepare_frame(converted,converted_meta,{'conversions':[{'column':'r','kind':'percent_to_decimal'}]}))
rejects(lambda:prepare_frame(prepared,metadata,{'derive':[{'operation':'pnl'}]}))
print('无隐式清洗、参数边界、错误单位及错配样本拒绝：通过')

无隐式清洗、参数边界、错误单位及错配样本拒绝：通过
